# Module 09 — Cell-Cell Communication Analysis

This notebook visualizes pre-computed cell-cell communication (CCC) results
from `scripts/09_communication.py`. LIANA (LIgand-receptor ANAlysis) is used
to infer ligand-receptor interactions between cell types, comparing healthy
and degenerated IVD tissue.

**Key findings:**
- **28,878 interactions** in healthy tissue
- **27,011 interactions** in degenerated tissue (reversed from expected increase)
- Differential interaction analysis reveals condition-specific signaling rewiring
- Pain-relevant ligand-receptor interactions identified in degenerated tissue

**Data sources:**
- `results/communication/interactions_healthy.tsv` — Healthy interactions
- `results/communication/interactions_degenerated.tsv` — Degenerated interactions
- `results/communication/differential_interactions.tsv` — Differential analysis
- `results/communication/pain_interactions.tsv` — Pain-relevant interactions
- `results/communication/interaction_plots/` — Visualization plots

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
RESULTS = BASE / 'results' / 'communication'
PLOT_DIR = RESULTS / 'interaction_plots'

print(f'Results directory: {RESULTS}')
print(f'Plot files:       {len(list(PLOT_DIR.glob("*.png")))} files')

## Interaction Counts Summary

Overview of the number of inferred ligand-receptor interactions in healthy
vs degenerated tissue.

In [ ]:
# Load interaction tables
healthy_path = RESULTS / 'interactions_healthy.tsv'
degen_path = RESULTS / 'interactions_degenerated.tsv'

if healthy_path.exists() and degen_path.exists():
    healthy = pd.read_csv(healthy_path, sep='\t')
    degen = pd.read_csv(degen_path, sep='\t')
    
    print(f'Healthy interactions:     {len(healthy):,}')
    print(f'Degenerated interactions: {len(degen):,}')
    print(f'Difference:              {len(degen) - len(healthy):+,}')
    print()
    
    # Unique source-target pairs
    if 'source' in healthy.columns and 'target' in healthy.columns:
        h_pairs = healthy.groupby(['source', 'target']).size()
        d_pairs = degen.groupby(['source', 'target']).size()
        print(f'Unique source-target pairs (healthy):     {len(h_pairs)}')
        print(f'Unique source-target pairs (degenerated): {len(d_pairs)}')
    
    # Preview
    cols = [c for c in ['source', 'target', 'ligand_complex', 'receptor_complex',
                        'lr_means', 'lrscore'] if c in healthy.columns]
    display(Markdown('**Top Healthy Interactions (by score)**'))
    display(healthy.nlargest(10, 'lrscore' if 'lrscore' in healthy.columns else cols[-1])[cols])
else:
    print('Interaction files not found')

## Interaction Heatmaps

Heatmaps showing the number/strength of interactions between each pair of
cell types, separately for healthy and degenerated conditions.

In [ ]:
# Healthy interaction heatmap
h_heatmap = PLOT_DIR / 'interaction_heatmap_healthy.png'
if h_heatmap.exists():
    display(Markdown('### Healthy — Cell-Cell Interaction Heatmap'))
    display(Image(filename=str(h_heatmap), width=800))
else:
    print('Healthy interaction heatmap not found')

In [ ]:
# Degenerated interaction heatmap
d_heatmap = PLOT_DIR / 'interaction_heatmap_degenerated.png'
if d_heatmap.exists():
    display(Markdown('### Degenerated — Cell-Cell Interaction Heatmap'))
    display(Image(filename=str(d_heatmap), width=800))
else:
    print('Degenerated interaction heatmap not found')

## Top Interactions by Condition

The highest-scoring ligand-receptor interactions in each condition.

In [ ]:
# Top interactions plots
for condition in ['healthy', 'degenerated']:
    top_path = PLOT_DIR / f'top_interactions_{condition}.png'
    if top_path.exists():
        display(Markdown(f'### Top Interactions — {condition.capitalize()}'))
        display(Image(filename=str(top_path), width=800))
    else:
        print(f'Top interactions plot not found for {condition}')

## Differential Interactions

Interactions that differ significantly between healthy and degenerated
conditions. Positive log fold change indicates stronger signaling in
degeneration; negative indicates loss of signaling.

In [ ]:
# Differential interactions table
diff_path = RESULTS / 'differential_interactions.tsv'
if diff_path.exists():
    diff = pd.read_csv(diff_path, sep='\t')
    print(f'Total differential interactions: {len(diff)}')
    
    # Show top gained and lost interactions
    if 'lr_logfc' in diff.columns:
        gained = diff.nlargest(15, 'lr_logfc')
        lost = diff.nsmallest(15, 'lr_logfc')
        cols = [c for c in ['source', 'target', 'ligand_complex', 'receptor_complex',
                            'lr_logfc', 'lr_means'] if c in diff.columns]
        display(Markdown('**Top 15 Gained in Degeneration (highest logFC)**'))
        display(gained[cols])
        display(Markdown('**Top 15 Lost in Degeneration (lowest logFC)**'))
        display(lost[cols])
    else:
        display(diff.head(20))
else:
    print('Differential interactions file not found')

In [ ]:
# Differential interactions plot
diff_plot = PLOT_DIR / 'differential_interactions.png'
if diff_plot.exists():
    display(Markdown('### Differential Interactions Plot'))
    display(Image(filename=str(diff_plot), width=800))
else:
    print('Differential interactions plot not found')

## Pain-Relevant Interactions

Ligand-receptor interactions involving pain-related molecules (nociceptive
signaling, inflammatory mediators, neurotrophins). These interactions are
particularly relevant for understanding discogenic pain.

In [ ]:
# Pain-relevant interactions
pain_path = RESULTS / 'pain_interactions.tsv'
if pain_path.exists():
    pain = pd.read_csv(pain_path, sep='\t')
    print(f'Total pain-relevant interactions: {len(pain)}')
    
    if 'condition' in pain.columns:
        for cond in pain['condition'].unique():
            n = len(pain[pain['condition'] == cond])
            print(f'  {cond}: {n} interactions')
    
    # Show pain interactions grouped by category
    if 'pain_categories' in pain.columns:
        print(f'\nPain categories represented:')
        # Pain categories may contain multiple categories per row
        all_cats = pain['pain_categories'].dropna().str.split(',').explode().str.strip()
        for cat, count in all_cats.value_counts().items():
            print(f'  {cat}: {count}')
    
    cols = [c for c in ['source', 'target', 'ligand_complex', 'receptor_complex',
                        'lr_means', 'lrscore', 'condition', 'pain_categories']
            if c in pain.columns]
    
    # Show degeneration-specific pain interactions
    if 'condition' in pain.columns:
        degen_pain = pain[pain['condition'] == 'degenerated']
        display(Markdown('**Pain-Relevant Interactions in Degenerated Tissue**'))
        display(degen_pain[cols].head(30).style.set_caption(
            'Top Pain-Relevant Interactions (Degenerated)'))
    else:
        display(pain[cols].head(30))
else:
    print('Pain interactions file not found')

## Status — Module 09 Complete

### Summary
- LIANA cell-cell communication analysis completed for healthy vs degenerated
- **28,878 healthy** / **27,011 degenerated** interactions inferred
- Differential analysis identifies rewired signaling pathways
- Pain-relevant interactions catalogued for clinical interpretation

### Key observations
- **Reversed pattern:** Fewer total interactions in degeneration (contrary to
  the naive expectation of increased inflammatory signaling)
- This may reflect loss of homeostatic cell-cell communication in degenerated
  tissue, with a shift toward fewer but more intense pro-inflammatory signals
- Pain-relevant interactions (e.g., CXCL8-SDC2, PTGS2-CAV1) are enriched in
  degenerated tissue
- Immune cells (macrophages, T cells) show altered interaction profiles with
  disc-resident cells in degeneration